# How LangChain Interacts with LLMs

## Overview

This notebook explains the core mechanisms of how LangChain interacts with Language Models, covering:

1. **Introduction** - Setting up LLM connections (cloud vs. local)
2. **Flexible Input Formats** - How LangChain accepts multiple input types
3. **Response Parsing** - How LangChain parses LLM outputs
4. **Provider Differences** - Why different endpoints require different parsing strategies
5. **Consistent Interface** - How LangChain achieves a unified API across providers

---

## 1. Introduction: Connecting to an LLM

Any agent framework needs to interact with an LLM. This can be achieved through two approaches:

### Option A: Cloud-based LLM Provider (e.g., OpenAI)
- Requires an account and API key
- Pay-per-token pricing
- **Not recommended during development** — bugs causing infinite loops can result in unexpected costs

### Option B: Local LLM Server (Recommended for beginners)
- Options: Hugging Face, Ollama, or LM Studio
- **LM Studio** is used in this notebook for its ease of use
- Provides a user-friendly interface to monitor requests sent to your LLM server

This notebook demonstrates LangChain with an LM Studio server. You can replace the URL and API key settings with your own provider's credentials.

In [38]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# Setup connection parameters
base_url = os.getenv("LMSTUDIO_BASE_URL", "http://localhost:1234/v1")
model_name = os.getenv("LMSTUDIO_MODEL", "openai/gpt-oss-20b")
api_key = os.getenv("LMSTUDIO_API_KEY", "lm-studio")

## 2. LangChain's Key Function: Input Reformatting and Output Parsing

LangChain provides **flexibility in input formats** and **consistent output parsing**. This is one of its core value propositions.

Check the offical doc for details https://docs.langchain.com/oss/python/langchain/models
you can find information such as stream batch etc there

### 2.1 Flexible Input Formats

LangChain accepts **different input formats** and converts them all to the provider's required format:

In [39]:
# Initialize model
model = ChatOpenAI(
    base_url=base_url,
    api_key=api_key,
    model=model_name,
)

print('LangChain accepts multiple input formats:\n')

# Format 1: Plain string
print('Format 1: Plain string')
reply1 = model.invoke('how are you')
print(f"Input: 'how are you'")
print(f"Output: {reply1.content}\n")
print('='*50)

# Format 2: Dictionary with role and content
print('\nFormat 2: Dictionary with role and content')
reply2 = model.invoke([{"role": "user", "content": "what is the weather in sf"}])
print(f"Input: [{{'role': 'user', 'content': 'what is the weather in sf'}}]")
print(f"Output: {reply2.content}\n")
print('='*50)

# Format 3: LangChain HumanMessage object
print('\nFormat 3: LangChain HumanMessage object')
reply3 = model.invoke([HumanMessage(content="how are you")])
print(f"Input: [HumanMessage(content='how are you')]")
print(f"Output: {reply3.content}\n")

LangChain accepts multiple input formats:

Format 1: Plain string
Input: 'how are you'
Output: I’m doing great—thanks for asking! How about you? Anything interesting on your end today?


Format 2: Dictionary with role and content
Input: [{'role': 'user', 'content': 'what is the weather in sf'}]
Output: I’m sorry, but I don’t have real‑time data access.  For the most accurate and up‑to‑date weather in San Francisco, please check a reliable source such as:

- **Weather apps** (e.g., Weather.com, AccuWeather, the NOAA app)
- **Local news websites** (e.g., KGO, ABC7 San Francisco)
- **Weather widgets** on your phone or computer

Those will give you current conditions, a short‑term forecast, and any alerts that might affect the area.


Format 3: LangChain HumanMessage object
Input: [HumanMessage(content='how are you')]
Output: I’m doing great—thanks for asking! How can I help you today?



### 2.2 Response Parsing

LangChain parses the provider's response into a **structured AIMessage object**. You can access the content using the `.content` attribute:

- **Raw response**: Contains metadata, token usage, and the full response structure
- **Parsed content**: Accessible via `.content` field for easy extraction

In [40]:
print('Full AIMessage object:')
print(reply1)
print('\n' + '='*50)
print('\nExtracted content only:')
print(reply1.content)

Full AIMessage object:
content='I’m doing great—thanks for asking! How about you? Anything interesting on your end today?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 70, 'total_tokens': 105, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'openai/gpt-oss-20b', 'id': 'chatcmpl-o6asj46xp8fe9yyedu5tb', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019c8b6b-d78a-7c40-b134-ae772aa1e065-0' usage_metadata={'input_tokens': 70, 'output_tokens': 35, 'total_tokens': 105, 'input_token_details': {}, 'output_token_details': {}}


Extracted content only:
I’m doing great—thanks for asking! How about you? Anything interesting on your end today?


---

## 3. Provider Differences: Why Parsing Strategies Matter

**Important**: LangChain only reformats inputs and parses outputs. When providers change their API endpoints or response formats, LangChain must adapt its parsing strategy.

### Example: OpenAI's New Response Endpoint

OpenAI introduced a new response endpoint (`responses/v1`) designed for **structured multi-part outputs**. Instead of returning one flat message, it returns **typed output items**:

```json
{
  "output": [
    {
      "type": "reasoning",
      "content": [{"type": "reasoning_text", "text": "..."}]
    },
    {
      "type": "text",
      "text": "Hello!"
    }
  ]
}
```

This means you need **different parsing logic** depending on which endpoint you use.

In [41]:
# Using the new responses/v1 endpoint
model_new = ChatOpenAI(
    base_url=base_url,
    api_key=api_key,
    model=model_name,
    output_version="responses/v1",  # New structured response format
)

reply_new = model_new.invoke('how are you')

print('The entire reply structure:')
print(reply_new)

print('\n' + '='*50)
print('\nThe .content field now contains structured blocks:')
print(reply_new.content)

print('\n' + '='*50)
print('\nTo extract just the text, use .text property:')
print(reply_new.text)

The entire reply structure:
content=[{'id': 'rs_oh14f2e4hddvy5zp0ghdm', 'summary': [], 'type': 'reasoning', 'content': [{'text': 'Need to respond politely.', 'type': 'reasoning_text'}], 'status': 'completed'}, {'type': 'text', 'text': 'I’m doing great—thanks for asking! How can I help you today?', 'annotations': [], 'id': 'msg_ngrhzzkvzqo73x2rz2avov'}] additional_kwargs={} response_metadata={'id': 'resp_c619cd3a5911ed98da0b2fbb1ad3e6a718d0a2bc167bf3b3', 'created_at': 1771865654.0, 'metadata': {}, 'model': 'openai/gpt-oss-20b', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-20b'} id='resp_c619cd3a5911ed98da0b2fbb1ad3e6a718d0a2bc167bf3b3' usage_metadata={'input_tokens': 70, 'output_tokens': 20, 'total_tokens': 90, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 5}}


The .content field now contains structured blocks:
[{'id': 'rs_oh14f2e4hddvy5zp0ghdm', 'summary': [], 'type':

### Custom Parsing for Structured Responses

When using structured endpoints, you may want to extract specific components like reasoning and answer separately:

In [42]:
def parse_structured_response(blocks):
    """
    Parse structured response blocks into reasoning and answer components.
    Useful for models that support chain-of-thought reasoning.
    """
    thinking_parts = []
    answer_parts = []

    for block in blocks:
        block_type = block.get("type")

        # Extract reasoning/thinking
        if block_type == "reasoning":
            for item in block.get("content", []):
                if item.get("type") == "reasoning_text":
                    thinking_parts.append(item.get("text", ""))

        # Extract answer text
        elif block_type == "text":
            answer_parts.append(block.get("text", ""))

    thinking = "\n".join(thinking_parts).strip()
    answer = "\n".join(answer_parts).strip()

    return thinking, answer

# Parse the structured response
think, answer = parse_structured_response(reply_new.content)
print('Model reasoning: ', think)
print('Model answer: ', answer)

Model reasoning:  Need to respond politely.
Model answer:  I’m doing great—thanks for asking! How can I help you today?


---

## 4. Consistent Interface: How LangChain Achieves Unified API

LangChain provides a unified interface across different providers through `init_chat_model()`, allowing you to switch between providers without changing your code.

In [43]:
from langchain.chat_models import init_chat_model

# Using init_chat_model - provider-agnostic initialization
print("Using init_chat_model:")
print("=" * 50)

model_init = init_chat_model(
    model="openai/gpt-oss-20b",
    model_provider="openai",
    base_url=base_url,
    api_key=api_key,
    model_kwargs={"output_version": "responses/v1"}
)

print(f"Model type: {type(model_init)}")
print(f"Note: Returns ChatOpenAI instance\n")

response = model_init.invoke([HumanMessage(content="What is 2+2?")])
print(f"Question: What is 2+2?")
print(f"Answer: {response.text}")
print("\nBoth init_chat_model() and ChatOpenAI() provide the same .invoke() interface!")

Using init_chat_model:
Model type: <class 'langchain_openai.chat_models.base.ChatOpenAI'>
Note: Returns ChatOpenAI instance



/var/folders/6d/ntnq3jy50xl7l7d4j0ytjjf40000gn/T/ipykernel_97287/320593719.py:7: UserWarning: Parameters {'output_version'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  model_init = init_chat_model(


Question: What is 2+2?
Answer: 4

Both init_chat_model() and ChatOpenAI() provide the same .invoke() interface!


## Under the Hood: How LangChain Maintains a Consistent Interface

LangChain provides a unified interface across different LLM providers through a two-layer architecture:

### 1. Provider-Specific Implementation Layer

Each LLM provider (OpenAI, Anthropic, etc.) has its own implementation file that handles:
- **Message conversion**: Transforms user input into the provider's required format
- **Response parsing**: Parses provider-specific responses into a consistent structure

This is why each provider has its own `.py` file in LangChain.

### 2. Abstraction Layer: `init_chat_model`

The `init_chat_model` function acts as a factory that:
- Determines which provider to use
- Imports the appropriate provider-specific class
- Returns a unified interface

**Example from `langchain/chat_models/base.py`:**

```python
def _init_chat_model_helper(
    model: str,
    *,
    model_provider: str | None = None,
    **kwargs: Any,
) -> BaseChatModel:
    model, model_provider = _parse_model(model, model_provider)
    
    if model_provider == "openai":
        _check_pkg("langchain_openai")
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=model, **kwargs)
    
    # ... other providers handled similarly
```

The main `init_chat_model` function orchestrates this:

```python
def init_chat_model(
    model: str | None = None,
    *,
    model_provider: str | None = None,
    configurable_fields: Literal["any"] | list[str] | tuple[str, ...] | None = None,
    config_prefix: str | None = None,
    **kwargs: Any,
) -> BaseChatModel | _ConfigurableModel:
    
    if not configurable_fields:
        return _init_chat_model_helper(
            cast("str", model),
            model_provider=model_provider,
            **kwargs,
        )
    
    # Handle configurable models for dynamic switching
    if model:
        kwargs["model"] = model
    if model_provider:
        kwargs["model_provider"] = model_provider
    
    return _ConfigurableModel(
        default_config=kwargs,
        config_prefix=config_prefix,
        configurable_fields=configurable_fields,
    )
```

**Key Benefit:** Whether you use `ChatOpenAI` directly or `init_chat_model`, you get the same interface with `.invoke()`, allowing easy switching between providers without changing your code.